# 02 — Customer Behavior, Repeat Purchase & Review Retention

This notebook documents the behavioral analysis completed before RFM.

**Practice focus:** grain, joins, repeat definitions, right-censoring, leakage, and association vs causation.


In [ ]:
customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

customer_orders["customer_unique_id"].isna().sum()


In [ ]:
orders_per_customer = (
    customer_orders
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

orders_per_customer.value_counts().sort_index()


Observed:
- 96,096 unique customers
- 93,099 placed exactly one order
- 2,997 placed more than one order
- naive repeat-customer rate ≈ 3.12%

This showed that frequency alone does not equal customer value.


In [ ]:
customer_orders = customer_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

customer_orders["purchase_number"] = (
    customer_orders.groupby("customer_unique_id").cumcount() + 1
)


A large share of second purchases occurred very quickly after the first purchase. That suggested some "repeat" orders may belong to the same shopping episode.

Working behavioral rule: a meaningful return is **any later purchase more than 7 days after the first purchase**.


In [ ]:
first_purchase = (
    customer_orders
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .min()
    .rename("first_purchase")
)

customer_orders = customer_orders.merge(
    first_purchase,
    on="customer_unique_id",
    how="left"
)

customer_orders["days_from_first"] = (
    customer_orders["order_purchase_timestamp"]
    - customer_orders["first_purchase"]
).dt.total_seconds() / 86400

repeat_target = (
    customer_orders
    .groupby("customer_unique_id")["days_from_first"]
    .apply(lambda x: (x > 7).any())
    .astype(int)
    .rename("repeat_after_7d")
)


## Right-censoring

Customers acquired near the dataset end do not have enough time to show a repeat purchase. They are removed from this specific target evaluation.


In [ ]:
dataset_end = customer_orders["order_purchase_timestamp"].max()

eligible_customers = first_purchase[
    first_purchase <= dataset_end - pd.Timedelta(days=7)
].index

repeat_target_eligible = repeat_target[
    repeat_target.index.isin(eligible_customers)
]


## Review cleaning

Some orders have multiple review records and `review_id` is not perfectly unique. For order-level analysis, one deterministic review record is retained using the latest `review_answer_timestamp`.


In [ ]:
order_reviews = pd.read_csv("../data_raw/olist_order_reviews_dataset.csv")

order_reviews["review_answer_timestamp"] = pd.to_datetime(
    order_reviews["review_answer_timestamp"]
)

reviews_one_per_order = (
    order_reviews
    .sort_values(["order_id", "review_answer_timestamp"])
    .drop_duplicates("order_id", keep="last")
)

reviews_one_per_order["order_id"].duplicated().sum()


In [ ]:
first_orders = (
    customer_orders
    .sort_values(["customer_unique_id", "order_purchase_timestamp"])
    .drop_duplicates("customer_unique_id", keep="first")
    [["customer_unique_id", "order_id", "order_purchase_timestamp"]]
)

first_order_analysis = first_orders.merge(
    reviews_one_per_order[["order_id", "review_score"]],
    on="order_id",
    how="left"
).merge(
    repeat_target_eligible,
    on="customer_unique_id",
    how="inner"
)

review_analysis = first_order_analysis.dropna(
    subset=["review_score"]
).copy()


In [ ]:
review_retention = (
    review_analysis
    .groupby("review_score")
    .agg(
        customers=("customer_unique_id", "nunique"),
        repeat_customers=("repeat_after_7d", "sum"),
        repeat_rate=("repeat_after_7d", "mean")
    )
    .reset_index()
)

review_retention["repeat_rate_pct"] = (
    review_retention["repeat_rate"] * 100
).round(2)

review_retention


### Observed result

| Review | Customers | Repeat customers | Repeat rate |
|---:|---:|---:|---:|
| 1 | 10,994 | 188 | 1.71% |
| 2 | 3,035 | 57 | 1.88% |
| 3 | 7,856 | 144 | 1.83% |
| 4 | 18,497 | 352 | 1.90% |
| 5 | 54,977 | 1,197 | 2.18% |

**Interpretation:** 5-star first-order customers show the highest observed repeat rate. The relationship is weak and not strictly monotonic, so it is reported as an **association**, not causation.
